# Poker ML Pipeline - Feature Engineering (PySpark)

Converts raw poker hand history into opponent-style features and model-ready rows.

Uses PySpark for scalable data processing, then converts to pandas only for equity calculations.

Prerequisites: Serverless compute attached, notebook 01 completed

In [ ]:
# 02-latest (2024-12-01) - Position features; profit calculation moved to 04
# treys is required for hand evaluation - install without dependencies to avoid conflicts
%pip install treys --no-deps -q

In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import time

print("=" * 80)
print("SP-2: Opponent Range Feature Engineering (PySpark)")
print("=" * 80)
start_time = time.time()

# Get or create Spark session (already available in Databricks)
spark = SparkSession.builder.getOrCreate()
print(f"Spark version: {spark.version}")

In [ ]:
# Configuration
# NOTE: Use /Workspace paths for single-node operations (pandas, reading input)
# Use Unity Catalog Volumes for distributed Spark writes

WORKSPACE_DIR = '/Workspace/Users/leo.lwakabamba@gmail.com/poker-ml-data/'

# Unity Catalog Volume path for Spark distributed writes
UC_VOLUME_DIR = '/Volumes/pokerml/default/data/'

# Input paths (from Workspace - single-node reads work here)
DATA_DIR = WORKSPACE_DIR
CHUNKS_DIR = DATA_DIR + 'chunks/'

# Output paths (to Unity Catalog Volume for Spark compatibility)
OUTPUT_DIR = UC_VOLUME_DIR + 'processed/'

# ============================================================================
# DATA SIZE CONFIGURATION
# Read from pipeline config file (UC Volume - persists across Python restarts)
#
# DEBUG_MODE: True = sample first, False = use all data
# SAMPLE_FRACTION: Only used if DEBUG_MODE=True
# MAX_ROWS: ALWAYS applied (caps number of hands)
# ============================================================================
import json

# UC Volume path (persists across restarts, unlike /tmp)
CONFIG_PATH = '/Volumes/pokerml/default/data/pipeline_config.json'

try:
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    DEBUG_MODE = config.get('debug_mode', True)
    SAMPLE_FRACTION = config.get('sample_fraction', 0.01)
    MAX_ROWS = config.get('max_rows', 10000)
    print(f"*** Loaded pipeline config from {CONFIG_PATH} ***")
    print(f"    DEBUG_MODE={DEBUG_MODE}, MAX_ROWS={MAX_ROWS:,}")
except FileNotFoundError:
    # Defaults for standalone execution
    DEBUG_MODE = True
    SAMPLE_FRACTION = 0.01
    MAX_ROWS = 10000
    print(f"*** Config not found at {CONFIG_PATH}, using defaults ***")
except Exception as e:
    # Defaults if anything else goes wrong
    DEBUG_MODE = True
    SAMPLE_FRACTION = 0.01
    MAX_ROWS = 10000
    print(f"*** Error reading config ({e}), using defaults ***")
# ============================================================================

MAX_EQUITY_SAMPLES = 2000
ROLLING_WINDOWS = [3, 5, 10]
RANDOM_SEED = 42

print(f"\nInput directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\n*** DATA SIZE SETTINGS ***")
print(f"    DEBUG_MODE: {DEBUG_MODE}")
print(f"    MAX_ROWS: {MAX_ROWS:,} hands (always applied)")
if DEBUG_MODE:
    print(f"    SAMPLE_FRACTION: {SAMPLE_FRACTION} ({SAMPLE_FRACTION*100:.0f}%)")
    print(f"\n>>> Will sample {SAMPLE_FRACTION*100:.0f}%, then cap at {MAX_ROWS:,} hands")
else:
    print(f"\n>>> No sampling, will cap at {MAX_ROWS:,} hands")

In [ ]:
# Load data using Spark (handles large files efficiently)
print("\n[1/8] Loading hand history data with Spark...")

def load_spark_table(name: str):
    """Load table from parquet chunks or single file using Spark."""
    import os
    chunks_path = f"{CHUNKS_DIR}phh_multi_{name}_chunk_*.parquet"
    single_path = f"{DATA_DIR}phh_multi_{name}.parquet"
    csv_path = f"{DATA_DIR}phh_multi_{name}.csv"
    
    # Try chunks first
    try:
        df = spark.read.parquet(chunks_path)
        print(f"   {name}: Loaded from chunks")
        return df
    except:
        pass
    
    # Try single parquet
    try:
        df = spark.read.parquet(single_path)
        print(f"   {name}: Loaded from parquet")
        return df
    except:
        pass
    
    # Fall back to CSV
    df = spark.read.csv(csv_path, header=True, inferSchema=True)
    print(f"   {name}: Loaded from CSV")
    return df

actions_df = load_spark_table('actions')
seats_df = load_spark_table('seats')
boards_df = load_spark_table('boards')
hands_df = load_spark_table('hands')
showdowns_df = load_spark_table('showdown')

# ============================================================================
# FIND HAND_IDS THAT EXIST IN ALL REQUIRED TABLES
# This ensures sampling will have consistent data across all tables
# ============================================================================
print("\n   Finding hand_ids present in all tables...")

hands_in_actions = actions_df.select('hand_id').distinct()
hands_in_seats = seats_df.select('hand_id').distinct()
hands_in_boards = boards_df.select('hand_id').distinct()
hands_in_hands = hands_df.select('hand_id').distinct()
# Only include hands with valid showdown cards (not '????')
hands_in_showdowns = showdowns_df.filter(F.col('cards') != '????').select('hand_id').distinct()

# Intersection of all tables
common_hands = (hands_in_actions
    .join(hands_in_seats, on='hand_id', how='inner')
    .join(hands_in_boards, on='hand_id', how='inner')
    .join(hands_in_hands, on='hand_id', how='inner')
    .join(hands_in_showdowns, on='hand_id', how='inner')
)

total_common_hands = common_hands.count()
print(f"   Hand IDs in actions: {hands_in_actions.count():,}")
print(f"   Hand IDs in seats: {hands_in_seats.count():,}")
print(f"   Hand IDs in boards: {boards_df.select('hand_id').distinct().count():,}")
print(f"   Hand IDs in hands: {hands_in_hands.count():,}")
print(f"   Hand IDs in showdowns (with cards): {hands_in_showdowns.count():,}")
print(f"   Hand IDs in ALL tables (intersection): {total_common_hands:,}")

# ============================================================================
# SELECT TRACE_HAND_ID FROM COMMON HANDS
# This hand exists in ALL tables and has valid showdown data
# ============================================================================
TRACE_HAND_ID = common_hands.first()['hand_id']
print(f"\n*** TRACE_HAND_ID selected: {TRACE_HAND_ID} (from common_hands intersection) ***")

# ============================================================================
# TRACE: RAW DATA BEFORE ANY PROCESSING
# ============================================================================
print(f"\n[TRACE - BEFORE SAMPLING] Raw data for {TRACE_HAND_ID}:")

print("\n[TRACE] RAW ACTIONS:")
actions_df.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('idx').select(
    'hand_id', 'idx', 'street', 'actor', 'action_type', 'amount'
).show(30, truncate=False)

print("\n[TRACE] RAW SEATS:")
seats_df.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('seat_no').show(10, truncate=False)

print("\n[TRACE] RAW SHOWDOWN:")
showdowns_df.filter(F.col('hand_id') == TRACE_HAND_ID).show(10, truncate=False)

print("\n[TRACE] ACTOR vs PLAYER_NAME COMPARISON:")
actors_before = [r['actor'] for r in actions_df.filter(
    (F.col('hand_id') == TRACE_HAND_ID) & (F.col('actor') != 'Dealer')
).select('actor').distinct().collect()]
players_before = [r['player_name'] for r in seats_df.filter(
    F.col('hand_id') == TRACE_HAND_ID
).select('player_name').distinct().collect()]

print(f"   Actors in actions: {sorted(actors_before)}")
print(f"   Players in seats:  {sorted(players_before)}")
print(f"   Sets equal: {set(actors_before) == set(players_before)}")
if set(actors_before) != set(players_before):
    print(f"   *** MISMATCH DETECTED ***")
    print(f"   IN ACTIONS ONLY: {set(actors_before) - set(players_before)}")
    print(f"   IN SEATS ONLY: {set(players_before) - set(actors_before)}")

# ============================================================================
# APPLY DATA SIZE LIMITS
# DEBUG_MODE=True: Sample first, then cap at MAX_ROWS
# DEBUG_MODE=False: No sampling, but still cap at MAX_ROWS
# ============================================================================
print(f"\n   Applying data size limits...")

# Start with common hands (exist in all tables)
if DEBUG_MODE:
    print(f"   Step 1: Sampling {SAMPLE_FRACTION*100:.0f}% of {total_common_hands:,} hands...")
    sampled_hands = common_hands.sample(fraction=SAMPLE_FRACTION, seed=RANDOM_SEED)
    after_sample = sampled_hands.count()
    print(f"           After sampling: {after_sample:,} hands")
else:
    print(f"   Step 1: No sampling (DEBUG_MODE=False)")
    sampled_hands = common_hands
    after_sample = total_common_hands

# ALWAYS apply MAX_ROWS cap
print(f"   Step 2: Applying MAX_ROWS cap of {MAX_ROWS:,}...")

# Make sure TRACE_HAND_ID is always included
trace_hand_df = spark.createDataFrame([(TRACE_HAND_ID,)], ['hand_id'])
sampled_hands = sampled_hands.union(trace_hand_df).distinct()

# Apply cap
if sampled_hands.count() > MAX_ROWS:
    # Keep trace hand, limit the rest
    other_hands = sampled_hands.filter(F.col('hand_id') != TRACE_HAND_ID).limit(MAX_ROWS - 1)
    sampled_hands = other_hands.union(trace_hand_df)
    print(f"           Capped to {MAX_ROWS:,} hands (including TRACE_HAND_ID)")
else:
    print(f"           No cap needed ({sampled_hands.count():,} <= {MAX_ROWS:,})")

final_hand_count = sampled_hands.count()
print(f"\n   Final hand count: {final_hand_count:,} hands")

# Filter ALL tables to the SAME sampled hand_ids
actions_df = actions_df.join(sampled_hands, on='hand_id', how='inner')
seats_df = seats_df.join(sampled_hands, on='hand_id', how='inner')
boards_df = boards_df.join(sampled_hands, on='hand_id', how='inner')
hands_df = hands_df.join(sampled_hands, on='hand_id', how='inner')
showdowns_df = showdowns_df.join(sampled_hands, on='hand_id', how='inner')

print(f"\n   After filtering:")
print(f"   Actions: {actions_df.count():,} rows")
print(f"   Seats: {seats_df.count():,} rows")
print(f"   Boards: {boards_df.count():,} rows")
print(f"   Hands: {hands_df.count():,} rows")
print(f"   Showdowns: {showdowns_df.count():,} rows")

# ============================================================================
# TRACE: VERIFY TRACE_HAND_ID AFTER SAMPLING
# ============================================================================
print(f"\n[TRACE - AFTER SAMPLING] Verifying {TRACE_HAND_ID} in all tables:")
print(f"   In actions_df: {actions_df.filter(F.col('hand_id') == TRACE_HAND_ID).count()} rows")
print(f"   In seats_df: {seats_df.filter(F.col('hand_id') == TRACE_HAND_ID).count()} rows")
print(f"   In boards_df: {boards_df.filter(F.col('hand_id') == TRACE_HAND_ID).count()} rows")
print(f"   In hands_df: {hands_df.filter(F.col('hand_id') == TRACE_HAND_ID).count()} rows")
print(f"   In showdowns_df: {showdowns_df.filter(F.col('hand_id') == TRACE_HAND_ID).count()} rows")

In [ ]:
# Street ordering for feature engineering
street_mapping = {'preflop': 0, 'flop': 1, 'turn': 2, 'river': 3, 'showdown': 4}
street_map_expr = F.create_map([F.lit(x) for item in street_mapping.items() for x in item])

In [ ]:
# Prepare action log with derived features using Spark
print("\n[2/8] Preparing action log with derived features...")

# Add derived columns
actions_prepared = actions_df.withColumn(
    'amount', F.col('amount').cast('float')
).fillna({'amount': 0.0}).withColumn(
    'street_rank', street_map_expr[F.col('street')]
).fillna({'street_rank': -1}).withColumn(
    'is_player_action', F.col('actor') != 'Dealer'
).withColumn(
    'is_raise', F.col('action_type') == 'bet_or_raise_to'
).withColumn(
    'is_call', F.col('action_type') == 'call_or_check'
).withColumn(
    'is_fold', F.col('action_type') == 'fold'
).withColumn(
    'is_voluntary', (F.col('is_player_action')) & (F.col('is_raise') | F.col('is_call'))
).withColumn(
    'pot_contribution', F.when(F.col('is_player_action'), F.col('amount')).otherwise(0.0)
)

# Compute cumulative pot size using window function
print("   - Computing cumulative pot size...")
window_pot = Window.partitionBy('hand_id').orderBy(F.monotonically_increasing_id())
actions_prepared = actions_prepared.withColumn(
    'pot_size', F.sum('pot_contribution').over(window_pot)
)

# ============================================================================
# FIXED: Assign hand_sequence for chronological ordering (used in rolling windows)
# This is NOT the poker hand strength - that will be computed separately as hand_rank
# ============================================================================
print("   - Assigning hand sequence numbers (for rolling window ordering)...")
hand_sequences = actions_prepared.select('hand_id').distinct().withColumn(
    'hand_sequence', F.monotonically_increasing_id() + 1
)
actions_prepared = actions_prepared.join(hand_sequences, on='hand_id', how='left')

# ============================================================================
# NOTE: Actual poker hand_rank (1-7462) will be computed in cell-11 after
# hole cards are available. We add a placeholder here.
# ============================================================================
actions_prepared = actions_prepared.withColumn('hand_rank', F.lit(None).cast('int'))

print(f"   Prepared {actions_prepared.count():,} actions")

# ============================================================================
# TRACE: actions_prepared for TRACE_HAND_ID
# ============================================================================
print(f"\n[TRACE] actions_prepared for {TRACE_HAND_ID}:")
actions_prepared.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('idx').select(
    'hand_id', 'idx', 'street', 'actor', 'action_type', 'amount', 
    'is_player_action', 'pot_size', 'hand_sequence'
).show(30, truncate=False)

In [ ]:
# Build hand-level summary per player using Spark
print("\n[3/8] Building hand-level summaries...")

# Filter to player actions
player_actions = actions_prepared.filter(F.col('is_player_action'))

# Create summary flags
player_actions = player_actions.withColumn(
    'preflop_flag', F.col('street') == 'preflop'
).withColumn(
    'voluntary_preflop', F.col('is_voluntary') & F.col('preflop_flag')
).withColumn(
    'preflop_raise', F.col('is_raise') & F.col('preflop_flag')
).withColumn(
    'saw_flop_flag', F.col('street_rank') >= 1
).withColumn(
    'saw_turn_flag', F.col('street_rank') >= 2
).withColumn(
    'saw_river_flag', F.col('street_rank') >= 3
).withColumn(
    'showdown_flag', F.col('street_rank') >= 4
)

# Aggregate by player and hand
# FIXED: Use hand_sequence for chronological ordering (not hand_rank which is poker strength)
print("   - Aggregating by player and hand...")
hand_summary = player_actions.groupBy('actor', 'hand_id').agg(
    F.max(F.col('voluntary_preflop').cast('int')).alias('voluntary_preflop'),
    F.max(F.col('preflop_raise').cast('int')).alias('preflop_raise'),
    F.max(F.col('saw_flop_flag').cast('int')).alias('saw_flop'),
    F.max(F.col('saw_turn_flag').cast('int')).alias('saw_turn'),
    F.max(F.col('saw_river_flag').cast('int')).alias('saw_river'),
    F.max(F.col('showdown_flag').cast('int')).alias('showdown'),
    F.sum(F.col('is_raise').cast('int')).alias('aggressive_actions'),
    F.sum(F.col('is_call').cast('int')).alias('call_actions'),
    F.max(F.col('is_fold').cast('int')).alias('folded_in_hand'),
    F.max('street_rank').alias('final_street_rank'),
    F.max('hand_sequence').alias('hand_sequence')  # FIXED: Use hand_sequence for rolling windows
)

# ============================================================================
# TRACE: hand_summary BEFORE position join
# ============================================================================
print(f"\n[TRACE] hand_summary BEFORE position join for {TRACE_HAND_ID}:")
hand_summary.filter(F.col('hand_id') == TRACE_HAND_ID).show(10, truncate=False)

# ============================================================================
# POSITION FEATURE COMPUTATION
# ============================================================================
print("   - Computing position features from seat data...")

# Check which columns exist in seats table
seat_columns = seats_df.columns
has_position = 'position_from_button' in seat_columns and 'position_name' in seat_columns

if has_position:
    print("   - Position features found in seats table (using directly)")
    seat_info = seats_df.select(
        F.col('hand_id'),
        F.col('player_name').alias('actor'),
        F.col('seat_no').cast('int'),
        F.col('starting_stack').cast('float'),
        F.col('position_from_button').cast('int'),
        F.col('position_name')
    )
    # Get num_players separately
    players_per_hand = seats_df.groupBy('hand_id').agg(
        F.count('*').alias('num_players')
    )
    seat_info = seat_info.join(players_per_hand, on='hand_id', how='left')
else:
    print("   - Position features not in seats table - computing from seat order...")
    
    # Get num_players per hand
    players_per_hand = seats_df.groupBy('hand_id').agg(
        F.count('*').alias('num_players')
    )
    
    # Add num_players to seats
    seats_with_count = seats_df.join(players_per_hand, on='hand_id', how='left')
    
    # Rank players by seat_no within each hand
    window_seat_rank = Window.partitionBy('hand_id').orderBy('seat_no')
    
    seats_ranked = seats_with_count.withColumn(
        'seat_rank', F.row_number().over(window_seat_rank)
    )
    
    # Map seat_rank to position_from_button
    seats_ranked = seats_ranked.withColumn(
        'position_from_button',
        F.when(F.col('seat_rank') == F.col('num_players'), 0)
         .otherwise(F.col('seat_rank'))
    )
    
    # Map position to name
    seats_ranked = seats_ranked.withColumn(
        'position_name',
        F.when(F.col('position_from_button') == 0, 'BTN')
         .when(F.col('position_from_button') == 1, 'SB')
         .when(F.col('position_from_button') == 2, 'BB')
         .when((F.col('num_players') <= 4) & (F.col('position_from_button') == 3), 'UTG')
         .when((F.col('num_players') == 5) & (F.col('position_from_button') == 3), 'UTG')
         .when((F.col('num_players') == 5) & (F.col('position_from_button') == 4), 'CO')
         .when((F.col('num_players') == 6) & (F.col('position_from_button') == 3), 'UTG')
         .when((F.col('num_players') == 6) & (F.col('position_from_button') == 4), 'MP')
         .when((F.col('num_players') == 6) & (F.col('position_from_button') == 5), 'CO')
         .when(F.col('position_from_button') == 3, 'UTG')
         .when(F.col('position_from_button') == 4, 'UTG+1')
         .when(F.col('position_from_button') == F.col('num_players') - 2, 'CO')
         .when(F.col('position_from_button') == F.col('num_players') - 3, 'HJ')
         .otherwise('MP')
    )
    
    seat_info = seats_ranked.select(
        F.col('hand_id'),
        F.col('player_name').alias('actor'),
        F.col('seat_no').cast('int'),
        F.col('starting_stack').cast('float'),
        F.col('position_from_button').cast('int'),
        F.col('position_name'),
        F.col('num_players').cast('int')
    )
    
    print("   - Position features computed successfully")

# ============================================================================
# TRACE: seat_info (computed positions)
# ============================================================================
print(f"\n[TRACE] seat_info (computed positions) for {TRACE_HAND_ID}:")
seat_info.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('seat_no').show(10, truncate=False)

# Deduplicate seat_info to ensure 1:1 join
seat_info = seat_info.dropDuplicates(['hand_id', 'actor'])

# Compute median stack per hand
median_stack = seats_df.groupBy('hand_id').agg(
    F.expr('percentile_approx(starting_stack, 0.5)').alias('median_stack')
)

# Join seat_info (with position) to hand_summary
pre_join_count = hand_summary.count()
hand_summary = hand_summary.join(seat_info, on=['hand_id', 'actor'], how='left')
hand_summary = hand_summary.join(median_stack, on='hand_id', how='left')

hand_summary = hand_summary.withColumn(
    'stack_vs_table_median', F.col('starting_stack') / F.col('median_stack')
)

post_join_count = hand_summary.count()
print(f"\n   Hand summaries: {post_join_count:,} rows")

# Show position distribution
print("   Position distribution:")
hand_summary.groupBy('position_name').count().orderBy('count', ascending=False).show()

# Check NULL rate
null_count = hand_summary.filter(F.col('position_name').isNull()).count()
non_null_count = hand_summary.filter(F.col('position_name').isNotNull()).count()
print(f"   Position NULL: {null_count:,} ({100*null_count/post_join_count:.1f}%)")
print(f"   Position filled: {non_null_count:,} ({100*non_null_count/post_join_count:.1f}%)")

# ============================================================================
# TRACE: hand_summary AFTER position join
# ============================================================================
print(f"\n[TRACE] hand_summary AFTER position join for {TRACE_HAND_ID}:")
hand_summary.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'position_from_button', 'position_name', 'num_players',
    'starting_stack', 'voluntary_preflop', 'preflop_raise'
).show(10, truncate=False)

In [ ]:
# Compute rolling opponent features using Spark window functions
print("\n[4/8] Computing rolling opponent features...")

# FIXED: Use hand_sequence for rolling window ordering (not hand_rank which is poker strength)
# Window for rolling calculations per actor - ordered by hand_sequence (chronological)
for w in ROLLING_WINDOWS:
    window_roll = Window.partitionBy('actor').orderBy('hand_sequence').rowsBetween(-w, -1)
    
    hand_summary = hand_summary.withColumn(
        f'vpip_last{w}_hist', F.avg('voluntary_preflop').over(window_roll)
    ).withColumn(
        f'pfr_last{w}_hist', F.avg('preflop_raise').over(window_roll)
    ).withColumn(
        f'street_adv_last{w}_hist', F.avg('saw_flop').over(window_roll)
    ).withColumn(
        f'agg_factor_last{w}_hist', 
        F.sum('aggressive_actions').over(window_roll) / (F.sum('call_actions').over(window_roll) + 0.000001)
    ).withColumn(
        f'stack_trend_last{w}_hist',
        F.col('starting_stack') - F.avg('starting_stack').over(window_roll)
    )
    print(f"   - Added window {w} features")

print(f"   Added rolling features for windows: {ROLLING_WINDOWS}")

In [ ]:
# Create event-level rows for modeling
print("\n[5/8] Creating event-level feature rows...")

# Get player events with action counts
# IMPORTANT: Filter out 'show' actions - these are showdown card reveals, NOT player decisions
# CRITICAL FIX: Use original idx from actions_df for ordering, NOT monotonically_increasing_id()

# The raw actions_df has 'idx' which is the correct action sequence within each hand
# We should preserve this and use it for ordering

player_events = actions_prepared.filter(
    (F.col('is_player_action')) & 
    (F.col('action_type') != 'show')  # Exclude showdown card reveals
)

# Verify idx exists and use it for ordering
if 'idx' in player_events.columns:
    print("   Using original 'idx' from raw actions (correct ordering)")
    # Rename idx to action_no_in_hand for consistency, keeping original ordering
    player_events = player_events.withColumnRenamed('idx', 'action_no_in_hand')
else:
    print("   WARNING: 'idx' not found - creating action_no_in_hand from street order")
    # Fallback: create sequence based on street order then row position
    street_order = F.create_map([F.lit(x) for item in street_mapping.items() for x in item])
    player_events = player_events.withColumn('_street_order', street_order[F.col('street')])
    
    window_action = Window.partitionBy('hand_id').orderBy('_street_order', F.monotonically_increasing_id())
    player_events = player_events.withColumn(
        'action_no_in_hand', F.row_number().over(window_action)
    ).drop('_street_order')

# Compute cumulative raise/call counts using the CORRECT ordering
window_cumulative = Window.partitionBy('hand_id').orderBy('action_no_in_hand')

player_events = player_events.withColumn(
    'raises_so_far', F.sum(F.col('is_raise').cast('int')).over(window_cumulative)
).withColumn(
    'calls_so_far', F.sum(F.col('is_call').cast('int')).over(window_cumulative)
)

# Street action number (action within street)
window_street = Window.partitionBy('hand_id', 'street').orderBy('action_no_in_hand')
player_events = player_events.withColumn(
    'street_action_no', F.row_number().over(window_street)
)

# Get history columns from hand_summary
history_cols = [c for c in hand_summary.columns if c.endswith('_hist')]

# Include position features in the merge
# NOTE: hand_rank is NOT in hand_summary - it will be computed later in cell-12
# Only include columns that actually exist in hand_summary
merge_cols = [
    'actor', 'hand_id', 'starting_stack', 'stack_vs_table_median',
    'position_from_button', 'position_name', 'num_players'  # Position features
] + history_cols

# Verify all columns exist before selecting
available_merge_cols = [c for c in merge_cols if c in hand_summary.columns]
missing_cols = [c for c in merge_cols if c not in hand_summary.columns]
if missing_cols:
    print(f"   WARNING: Columns not in hand_summary: {missing_cols}")

summary_for_merge = hand_summary.select(available_merge_cols).dropDuplicates(['actor', 'hand_id'])

# ============================================================================
# TRACE: Verify action ordering is correct
# ============================================================================
print(f"\n[TRACE] Verifying action order for {TRACE_HAND_ID}:")
player_events.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('action_no_in_hand').select(
    'hand_id', 'action_no_in_hand', 'street', 'actor', 'action_type', 'amount'
).show(30, truncate=False)

# Verify street ordering is correct
print(f"\n[DEBUG] Checking if actions are in correct street order...")
street_order_check = player_events.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'action_no_in_hand', 'street'
).orderBy('action_no_in_hand').collect()

prev_street_rank = -1
order_correct = True
for row in street_order_check:
    curr_rank = street_mapping.get(row['street'], 99)
    if curr_rank < prev_street_rank:
        order_correct = False
        print(f"   ERROR: Street went backwards at action {row['action_no_in_hand']}: {row['street']}")
    prev_street_rank = curr_rank

if order_correct:
    print("   ✓ Street ordering is correct (preflop → flop → turn → river)")
else:
    print("   *** WARNING: Street ordering issue detected! ***")

# ============================================================================
# TRACE: summary_for_merge before join
# ============================================================================
print(f"\n[TRACE] summary_for_merge for {TRACE_HAND_ID}:")
summary_for_merge.filter(F.col('hand_id') == TRACE_HAND_ID).show(10, truncate=False)

# Prepare boards
boards_expanded = boards_df.fillna({'flop': '', 'turn': '', 'river': ''})

# ============================================================================
# Join summary features to player_events
# ============================================================================
player_events = player_events.join(
    summary_for_merge,
    on=['actor', 'hand_id'], 
    how='left'
).join(
    boards_expanded, on='hand_id', how='left'
)

print(f"   Event-level rows: {player_events.count():,}")

# Show position feature availability
if 'position_name' in player_events.columns:
    non_null_positions = player_events.filter(F.col('position_name').isNotNull()).count()
    total_positions = player_events.count()
    null_positions = total_positions - non_null_positions
    print(f"   Events with position data: {non_null_positions:,}")
    print(f"   Events missing position: {null_positions:,}")

# ============================================================================
# TRACE: player_events after all joins
# ============================================================================
print(f"\n[TRACE] player_events for {TRACE_HAND_ID}:")
player_events.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('action_no_in_hand').select(
    'hand_id', 'actor', 'street', 'action_type', 'action_no_in_hand',
    'position_from_button', 'position_name', 'num_players', 'starting_stack'
).show(30, truncate=False)

In [ ]:
# Register equity calculation functions as Spark UDFs
print("\n[6/8] Registering equity calculation UDFs for distributed processing...")

from pyspark.sql.functions import udf, col, array, struct
from pyspark.sql.types import FloatType, StringType, ArrayType, IntegerType

# Constants
RANK_VALUE = {'2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, 'T': 10, 'J': 11, 'Q': 12, 'K': 13, 'A': 14}
RANKS = '23456789TJQKA'
SUITS = 'cdhs'

def normalize_card(card):
    if not isinstance(card, str):
        return ''
    card = card.strip()
    if len(card) != 2 or '?' in card:
        return ''
    rank = card[0].upper()
    suit = card[1].lower()
    if rank not in RANK_VALUE or suit not in SUITS:
        return ''
    return f'{rank}{suit}'

def chunk_cards(card_str):
    if not isinstance(card_str, str):
        return []
    card_str = card_str.strip()
    cards = []
    for i in range(0, len(card_str), 2):
        normalized = normalize_card(card_str[i:i+2])
        if normalized:
            cards.append(normalized)
    return cards

def board_cards_for_street(street, flop, turn, river):
    flop_cards = chunk_cards(flop) if flop else []
    turn_card = normalize_card(turn) if turn else ''
    river_card = normalize_card(river) if river else ''

    if street == 'preflop':
        return []
    cards = []
    if street in ('flop', 'turn', 'river', 'showdown'):
        cards.extend(flop_cards)
    if street in ('turn', 'river', 'showdown') and turn_card:
        cards.append(turn_card)
    if street in ('river', 'showdown') and river_card:
        cards.append(river_card)
    return cards

def has_flush_draw(board_cards, hole_cards):
    if len(board_cards) < 3:
        return False
    board_suits = [c[1] for c in board_cards if len(c) == 2]
    hole_suits = [c[1] for c in hole_cards if len(c) == 2]
    for suit in SUITS:
        if board_suits.count(suit) >= 2 and hole_suits.count(suit) >= 1:
            if 4 <= board_suits.count(suit) + hole_suits.count(suit) < 5:
                return True
    return False

def has_four_to_straight(board_cards, hole_cards):
    if len(board_cards) < 3:
        return False
    ranks = set()
    for card in board_cards + hole_cards:
        if len(card) != 2:
            continue
        val = RANK_VALUE.get(card[0])
        if val:
            ranks.add(val)
            if val == 14:
                ranks.add(1)
    for start in range(1, 11):
        span = set(range(start, start + 5))
        if 4 <= len(ranks & span) < 5:
            return True
    return False

def compute_hand_strength(hole_cards, board_cards):
    """Compute approximate hand strength without external library."""
    if len(hole_cards) != 2:
        return None

    all_cards = hole_cards + board_cards
    if len(all_cards) < 2:
        return 0.5

    ranks = [RANK_VALUE.get(c[0], 0) for c in all_cards if len(c) == 2]
    suits = [c[1] for c in all_cards if len(c) == 2]

    if not ranks:
        return 0.5

    rank_counts = {}
    for r in ranks:
        rank_counts[r] = rank_counts.get(r, 0) + 1

    max_count = max(rank_counts.values()) if rank_counts else 0
    pairs = sum(1 for c in rank_counts.values() if c == 2)

    suit_counts = {}
    for s in suits:
        suit_counts[s] = suit_counts.get(s, 0) + 1
    has_flush = max(suit_counts.values()) >= 5 if suit_counts else False

    unique_ranks = sorted(set(ranks))
    if 14 in unique_ranks:
        unique_ranks = [1] + unique_ranks
    has_straight = False
    for i in range(len(unique_ranks) - 4):
        if unique_ranks[i+4] - unique_ranks[i] == 4:
            has_straight = True
            break

    strength = 0.2
    hole_ranks = [RANK_VALUE.get(c[0], 0) for c in hole_cards if len(c) == 2]
    high_card_bonus = sum(hole_ranks) / 28.0 * 0.15
    strength += high_card_bonus

    if max_count >= 4:
        strength = 0.95
    elif max_count == 3 and pairs >= 1:
        strength = 0.90
    elif has_flush:
        strength = 0.85
    elif has_straight:
        strength = 0.80
    elif max_count == 3:
        strength = 0.70
    elif pairs >= 2:
        strength = 0.60
    elif pairs == 1:
        strength = 0.45

    if strength < 0.5 and len(board_cards) >= 3:
        if has_flush_draw(board_cards, hole_cards):
            strength = max(strength, 0.40)
        if has_four_to_straight(board_cards, hole_cards):
            strength = max(strength, 0.38)

    return min(1.0, strength)

def bucket_from_strength(strength, street, board_cards, hole_cards):
    if strength is None:
        return None
    if strength >= 0.8:
        return 'nutted'
    if strength >= 0.65:
        return 'strong'
    if strength >= 0.4:
        return 'medium'
    if street != 'river' and (has_flush_draw(board_cards, hole_cards) or has_four_to_straight(board_cards, hole_cards)):
        return 'draw'
    return 'air'

# ============================================================================
# NEW: Compute actual poker hand_rank (1-7462) using treys library
# This is the CORRECT hand_rank for model features (lower = better hand)
# ============================================================================
def compute_hand_rank_treys(hole_cards, board_cards):
    """
    Compute poker hand rank using treys evaluator.
    Returns 1 (best, royal flush) to 7462 (worst, 7-high).
    
    For preflop (no board), returns an approximation based on hole card strength.
    """
    if not hole_cards or len(hole_cards) != 2:
        return None
    
    try:
        from treys import Card, Evaluator
        evaluator = Evaluator()
        
        # Convert cards to treys format
        def to_treys_card(card_str):
            """Convert 'Ah' format to treys Card int."""
            if not card_str or len(card_str) != 2:
                return None
            rank = card_str[0].upper()
            suit = card_str[1].lower()
            return Card.new(rank + suit)
        
        hole = [to_treys_card(c) for c in hole_cards]
        if None in hole:
            return None
        
        # Postflop: use actual treys evaluation
        if board_cards and len(board_cards) >= 3:
            board = [to_treys_card(c) for c in board_cards[:5]]  # Max 5 board cards
            if None in board:
                return None
            return evaluator.evaluate(board, hole)
        
        # Preflop: approximate rank based on hole card strength
        # Use a formula that maps to approximate hand rank distribution
        r1 = RANK_VALUE.get(hole_cards[0][0], 0)
        r2 = RANK_VALUE.get(hole_cards[1][0], 0)
        high = max(r1, r2)
        low = min(r1, r2)
        paired = 1 if r1 == r2 else 0
        suited = 1 if len(hole_cards[0]) == 2 and len(hole_cards[1]) == 2 and hole_cards[0][1] == hole_cards[1][1] else 0
        
        # Approximate preflop rank (lower = better)
        # AA = ~100, 72o = ~7000
        if paired:
            # Pairs: AA=100, KK=200, ..., 22=1300
            rank = 100 + (14 - high) * 100
        else:
            # Non-pairs: base on high card + kicker + suitedness
            base = 1400 + (14 - high) * 400 + (14 - low) * 20
            if suited:
                rank = base - 200  # Suited bonus
            else:
                rank = base
        
        return int(min(7462, max(1, rank)))
        
    except ImportError:
        # Fallback if treys not available
        return compute_hand_rank_fallback(hole_cards, board_cards)
    except Exception as e:
        # Any other error, use fallback
        return compute_hand_rank_fallback(hole_cards, board_cards)

def compute_hand_rank_fallback(hole_cards, board_cards):
    """Fallback hand rank calculation without treys."""
    if not hole_cards or len(hole_cards) != 2:
        return None
    
    r1 = RANK_VALUE.get(hole_cards[0][0], 0) if len(hole_cards[0]) >= 1 else 0
    r2 = RANK_VALUE.get(hole_cards[1][0], 0) if len(hole_cards[1]) >= 1 else 0
    high = max(r1, r2)
    low = min(r1, r2)
    paired = 1 if r1 == r2 else 0
    suited = 1 if len(hole_cards[0]) == 2 and len(hole_cards[1]) == 2 and hole_cards[0][1] == hole_cards[1][1] else 0
    
    if paired:
        rank = 100 + (14 - high) * 100
    else:
        base = 1400 + (14 - high) * 400 + (14 - low) * 20
        rank = base - 200 if suited else base
    
    return int(min(7462, max(1, rank)))

# Register UDFs
@udf(returnType=ArrayType(StringType()))
def udf_chunk_cards(card_str):
    return chunk_cards(card_str)

@udf(returnType=ArrayType(StringType()))
def udf_board_cards(street, flop, turn, river):
    return board_cards_for_street(street, flop, turn, river)

@udf(returnType=FloatType())
def udf_hand_strength(hole_cards, board_cards):
    if hole_cards is None or len(hole_cards) != 2:
        return None
    return compute_hand_strength(list(hole_cards), list(board_cards) if board_cards else [])

# NEW: UDF for actual poker hand_rank (1-7462)
@udf(returnType=IntegerType())
def udf_hand_rank(hole_cards, board_cards):
    """Compute actual poker hand rank (1=best, 7462=worst)."""
    if hole_cards is None or len(hole_cards) != 2:
        return None
    return compute_hand_rank_treys(list(hole_cards), list(board_cards) if board_cards else [])

@udf(returnType=StringType())
def udf_bucket(strength, street, board_cards, hole_cards):
    if strength is None:
        return None
    bc = list(board_cards) if board_cards else []
    hc = list(hole_cards) if hole_cards else []
    return bucket_from_strength(strength, street, bc, hc)

@udf(returnType=IntegerType())
def udf_board_paired(board_cards):
    if not board_cards or len(board_cards) < 3:
        return 0
    ranks = [c[0] for c in board_cards if len(c) == 2]
    return int(len(ranks) != len(set(ranks)))

@udf(returnType=IntegerType())
def udf_flush_possible(board_cards):
    if not board_cards or len(board_cards) < 3:
        return 0
    suits = [c[1] for c in board_cards if len(c) == 2]
    return int(any(suits.count(s) >= 3 for s in 'cdhs'))

@udf(returnType=IntegerType())
def udf_straight_possible(board_cards):
    if not board_cards or len(board_cards) < 3:
        return 0
    ranks = sorted({RANK_VALUE.get(c[0], 0) for c in board_cards if len(c) == 2 and c[0] in RANK_VALUE})
    if not ranks:
        return 0
    for i in range(len(ranks)):
        window = ranks[i:i+3]
        if len(window) >= 3 and window[-1] - window[0] <= 4:
            return 1
    return 0

@udf(returnType=IntegerType())
def udf_hole_pair(hole_cards):
    if not hole_cards or len(hole_cards) != 2:
        return 0
    return int(hole_cards[0][0] == hole_cards[1][0])

@udf(returnType=IntegerType())
def udf_flush_draw(board_cards, hole_cards):
    if not board_cards or not hole_cards:
        return 0
    return int(has_flush_draw(list(board_cards), list(hole_cards)))

@udf(returnType=IntegerType())
def udf_straight_draw(board_cards, hole_cards):
    if not board_cards or not hole_cards:
        return 0
    return int(has_four_to_straight(list(board_cards), list(hole_cards)))

print("   UDFs registered successfully")
print("   NEW: udf_hand_rank computes actual poker hand rank (1-7462)")

In [ ]:
# Create labels from showdown hands using distributed Spark UDFs
# ============================================================================
# FIX FOR ISSUE #6: Use LEFT JOIN to preserve fold actions
# Players who fold don't show cards, so INNER JOIN dropped them entirely
# Now we keep ALL player actions and add hole_cards where available
# ============================================================================
print("\n[7/8] Creating labels from showdown hands (distributed)...")

# Filter showdowns with known cards
showdown_cards = showdowns_df.filter(
    F.col('cards') != '????'
).select(
    F.col('hand_id'),
    F.col('player_name').alias('actor'),
    F.col('cards').alias('hole_cards_raw')
)

# Find hands with at least 2 players showing cards
hand_visibility = showdown_cards.groupBy('hand_id').agg(
    F.countDistinct('actor').alias('players_showing')
).filter(F.col('players_showing') >= 2)

showdown_cards_filtered = showdown_cards.join(
    hand_visibility.select('hand_id'), on='hand_id', how='inner'
)

hands_with_full_info_count = hand_visibility.count()
print(f"   Hands with full showdown info: {hands_with_full_info_count:,}")

# ============================================================================
# TRACE: showdown_cards for TRACE_HAND_ID
# ============================================================================
print(f"\n[TRACE] showdown_cards for {TRACE_HAND_ID}:")
showdown_cards.filter(F.col('hand_id') == TRACE_HAND_ID).show(10, truncate=False)

# Join player events with showdown info
# Keep only hands that went to showdown (need outcome data)
player_events_visible = player_events.join(
    hand_visibility.select('hand_id'), on='hand_id', how='inner'
)

# ============================================================================
# FIX: Use LEFT JOIN instead of INNER JOIN
# This preserves ALL player actions, including those who folded
# Players who folded will have NULL hole_cards_raw (unknown cards)
# ============================================================================
print("   Using LEFT JOIN to preserve fold actions...")
pre_join_count = player_events_visible.count()

player_events_labeled = player_events_visible.join(
    showdown_cards_filtered, on=['hand_id', 'actor'], how='left'  # CHANGED FROM 'inner' to 'left'
)

post_join_count = player_events_labeled.count()
print(f"   Before LEFT JOIN: {pre_join_count:,} rows")
print(f"   After LEFT JOIN: {post_join_count:,} rows")

# Count how many rows have known vs unknown hole cards
known_cards = player_events_labeled.filter(F.col('hole_cards_raw').isNotNull()).count()
unknown_cards = player_events_labeled.filter(F.col('hole_cards_raw').isNull()).count()
print(f"   Rows with known hole cards: {known_cards:,} ({100*known_cards/post_join_count:.1f}%)")
print(f"   Rows with unknown hole cards (folders): {unknown_cards:,} ({100*unknown_cards/post_join_count:.1f}%)")

# ============================================================================
# TRACE: Check action_type distribution to verify folds are preserved
# ============================================================================
print(f"\n[TRACE] Action type distribution AFTER LEFT JOIN:")
player_events_labeled.groupBy('action_type').count().orderBy('count', ascending=False).show()

# ============================================================================
# TRACE: player_events_labeled BEFORE card parsing
# ============================================================================
print(f"\n[TRACE] player_events_labeled BEFORE card parsing for {TRACE_HAND_ID}:")
player_events_labeled.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('action_no_in_hand').select(
    'hand_id', 'actor', 'street', 'action_type',
    'position_from_button', 'position_name', 'hole_cards_raw'
).show(20, truncate=False)

# Add card parsing columns using UDFs
# For players who folded (NULL hole_cards_raw), these will produce NULL/empty values
print("   Parsing cards (distributed)...")
player_events_labeled = player_events_labeled.withColumn(
    'hole_cards', 
    F.when(F.col('hole_cards_raw').isNotNull(), udf_chunk_cards(F.col('hole_cards_raw')))
    .otherwise(F.array())  # Empty array for unknown cards
).withColumn(
    'board_cards', udf_board_cards(F.col('street'), F.col('flop'), F.col('turn'), F.col('river'))
)

# Compute hand strength using distributed UDF
# For players who folded, hand_equity will be NULL (unknown)
print("   Computing hand strength (distributed across cluster)...")
player_events_labeled = player_events_labeled.withColumn(
    'hand_equity', 
    F.when(
        F.col('hole_cards_raw').isNotNull(),
        udf_hand_strength(F.col('hole_cards'), F.col('board_cards'))
    ).otherwise(F.lit(None).cast('float'))  # NULL equity for folders (we don't know their cards)
)

# ============================================================================
# DO NOT filter out NULLs - keep fold actions even without hand_equity
# The downstream notebooks can filter if needed for specific analyses
# ============================================================================
# REMOVED: player_events_labeled = player_events_labeled.filter(F.col('hand_strength').isNotNull())

# ============================================================================
# Assign bucket labels
# For folders: Assume 'air' - if they had a strong hand, they wouldn't have folded
# This is a reasonable heuristic for opponent modeling
# ============================================================================
print("   Assigning bucket labels...")
player_events_labeled = player_events_labeled.withColumn(
    'bucket_label', 
    F.when(
        F.col('hand_equity').isNotNull(),
        udf_bucket(F.col('hand_equity'), F.col('street'), F.col('board_cards'), F.col('hole_cards'))
    ).otherwise(F.lit('air'))  # Folders assumed to be 'air' for opponent modeling
)

total_labeled = player_events_labeled.count()
with_equity = player_events_labeled.filter(F.col('hand_equity').isNotNull()).count()
without_equity = player_events_labeled.filter(F.col('hand_equity').isNull()).count()
print(f"   Total actions: {total_labeled:,}")
print(f"   With hand_equity (showdown players): {with_equity:,}")
print(f"   Without hand_equity (folders, assumed air): {without_equity:,}")

# ============================================================================
# TRACE: player_events_labeled AFTER labeling
# ============================================================================
print(f"\n[TRACE] player_events_labeled AFTER labeling for {TRACE_HAND_ID}:")
player_events_labeled.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('action_no_in_hand').select(
    'hand_id', 'actor', 'street', 'action_type',
    'position_from_button', 'position_name', 'num_players',
    'hand_equity', 'bucket_label'
).show(20, truncate=False)

# ============================================================================
# Show fold actions specifically - should now have bucket_label = 'air'
# ============================================================================
print(f"\n[TRACE] FOLD actions in {TRACE_HAND_ID} (should show bucket_label='air'):")
player_events_labeled.filter(
    (F.col('hand_id') == TRACE_HAND_ID) & (F.col('action_type') == 'fold')
).select(
    'hand_id', 'actor', 'street', 'action_type', 'hand_equity', 'bucket_label'
).show(10, truncate=False)

In [ ]:
# Add board texture and hand strength features (distributed)
print("\n[8/8] Adding board texture features (distributed)...")

# Board texture features
player_events_labeled = player_events_labeled.withColumn(
    'board_pair_or_better', udf_board_paired(F.col('board_cards'))
).withColumn(
    'board_flush_possible', udf_flush_possible(F.col('board_cards'))
).withColumn(
    'board_straight_possible', udf_straight_possible(F.col('board_cards'))
)

# Hole card features
player_events_labeled = player_events_labeled.withColumn(
    'hole_pair_flag', udf_hole_pair(F.col('hole_cards'))
).withColumn(
    'has_flush_draw_flag', udf_flush_draw(F.col('board_cards'), F.col('hole_cards'))
).withColumn(
    'has_straight_draw_flag', udf_straight_draw(F.col('board_cards'), F.col('hole_cards'))
)

# Bet sizing
player_events_labeled = player_events_labeled.withColumn(
    'bet_pct_pot', F.when(F.col('pot_size') > 0, F.col('amount') / F.col('pot_size')).otherwise(0.0)
)

# ============================================================================
# NEW: Compute actual poker hand_rank (1-7462) using treys
# This is the CORRECT hand_rank for profit model features
# Lower rank = better hand (1=royal flush, 7462=7-high)
# ============================================================================
print("   Computing hand_rank (1-7462) using treys...")
player_events_labeled = player_events_labeled.withColumn(
    'hand_rank',
    F.when(
        F.col('hole_cards_raw').isNotNull(),
        udf_hand_rank(F.col('hole_cards'), F.col('board_cards'))
    ).otherwise(F.lit(None).cast('int'))
)

# Validate hand_rank values
hand_rank_stats = player_events_labeled.filter(
    F.col('hand_rank').isNotNull()
).select(
    F.min('hand_rank').alias('min_rank'),
    F.max('hand_rank').alias('max_rank'),
    F.avg('hand_rank').alias('avg_rank')
).collect()[0]

print(f"   hand_rank stats: min={hand_rank_stats['min_rank']}, max={hand_rank_stats['max_rank']}, avg={hand_rank_stats['avg_rank']:.0f}")
print(f"   ✓ hand_rank should be 1-7462 (lower = better hand)")

print("   Features complete")

In [ ]:
# Save results directly from Spark (no pandas conversion)
print("[9/9] Saving results...")

# Select columns for output - INCLUDING POSITION FEATURES (profit computed in notebook 04)
# FIX FOR ISSUE #3: Added 'hole_cards' to output (was computed but not saved)
output_columns = [
    'hand_id', 'actor', 'street', 'action_type', 'amount', 'pot_size',
    'hand_rank', 'starting_stack', 'stack_vs_table_median',
    'position_from_button', 'position_name', 'num_players',  # Position features
    'hand_equity', 'bucket_label', 'hole_cards',  # Added hole_cards (Issue #3 fix)
    'board_pair_or_better', 'board_flush_possible', 'board_straight_possible',
    'hole_pair_flag', 'has_flush_draw_flag', 'has_straight_draw_flag',
    'bet_pct_pot', 'action_no_in_hand', 'raises_so_far', 'calls_so_far'
]

# Add history columns
history_cols = [c for c in player_events_labeled.columns if c.endswith('_hist')]
output_columns.extend(history_cols)

# Filter to existing columns only
existing_cols = [c for c in output_columns if c in player_events_labeled.columns]
output_df = player_events_labeled.select(existing_cols)

# Count rows
total_rows = output_df.count()
print(f"   Total rows: {total_rows:,}")

# Check position feature availability
if 'position_name' in existing_cols:
    position_coverage = output_df.filter(F.col('position_name').isNotNull()).count()
    print(f"   Rows with position data: {position_coverage:,} ({100*position_coverage/total_rows:.1f}%)")
else:
    print("   WARNING: Position features not included (re-run notebook 01 first)")

# ============================================================================
# Check action type distribution in final output
# ============================================================================
print(f"\n[VALIDATION] Action type distribution in FINAL OUTPUT:")
output_df.groupBy('action_type').count().orderBy('count', ascending=False).show()

fold_count = output_df.filter(F.col('action_type') == 'fold').count()
if fold_count > 0:
    print(f"   ✓ Fold actions preserved: {fold_count:,}")
else:
    print(f"   *** WARNING: No fold actions in output! ***")

# ============================================================================
# TRACE: FINAL OUTPUT for TRACE_HAND_ID
# ============================================================================
print(f"\n[TRACE] FINAL OUTPUT for {TRACE_HAND_ID}:")
output_df.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('action_no_in_hand').show(30, truncate=False)

# Save as Parquet directly from Spark to DBFS (required for distributed writes)
PARQUET_PATH = OUTPUT_DIR + 'sp2_player_events_labeled'

print(f"   Saving to Parquet (partitioned) at DBFS...")
output_df.write.mode('overwrite').parquet(PARQUET_PATH)
print(f"   Saved: {PARQUET_PATH}")

# Also save a small sample as CSV for quick inspection (Workspace path OK for single-node)
SAMPLE_CSV_PATH = DATA_DIR + 'processed/sp2_player_events_sample.csv'
sample_df = output_df.limit(10000).toPandas()
sample_df.to_csv(SAMPLE_CSV_PATH, index=False)
print(f"   Saved sample: {SAMPLE_CSV_PATH} ({len(sample_df):,} rows)")

elapsed = time.time() - start_time
print("\n" + "=" * 80)
print("SP-2 Feature Engineering COMPLETE!")
print("=" * 80)
print(f"Runtime: {elapsed/60:.1f} minutes")
print(f"Labeled actions: {total_rows:,}")
print(f"\nOutput columns ({len(existing_cols)}):")
for col in sorted(existing_cols):
    print(f"   - {col}")
print(f"\nFiles created:")
print(f"   - {PARQUET_PATH}/ (Parquet, full dataset)")
print(f"   - {SAMPLE_CSV_PATH} (CSV sample for inspection)")
print(f"\n[SUCCESS] Features ready for opponent modeling (SP-3)!")